# 香港特区`CBERS-04`卫星`P10`影像重投影

## 程序包初始化

### 内建程序包

In [1]:
import io, sys, os; 
try:
    nb_dir = nb_dir; 
except NameError: 
    nb_dir = os.getcwd(); 
    sys.path.append(nb_dir); 

In [2]:
import re; 
import collections as coll, itertools as it; 
from __future__ import print_function; 

In [3]:
from datetime import datetime; 

In [4]:
import sqlite3; 

### 第三方和自定义程序包

In [5]:
import numpy as np, pandas as pd; 

In [6]:
_ = sys.stdout; 
with io.BytesIO() as sys.stdout: import idlpy; 
sys.stdout = _; 

In [7]:
py_pkg_dir = os.path.normpath(os.path.join(
    nb_dir, os.pardir, os.pardir, os.pardir, 
    "Python_Package"
) ); 
sys.path.append(py_pkg_dir); 

In [8]:
import cresda_metadata_parse as cresda; 
cresda = cresda.importlib.reload(cresda); 

In [9]:
idl_pkg_dir = os.path.normpath(os.path.join(
    nb_dir, os.pardir, os.pardir, os.pardir, "IDL_Package"
) ); 

In [10]:
import cresda_metadata_parse as cresda; 
cresda = cresda.importlib.reload(cresda); 

In [11]:
idl_cb04 = os.path.normpath(
    os.path.join(idl_pkg_dir, "CB04_P10_Preproc")
); 
idl_cb04_subpkg = (
    "metadata", 
); 
idl_cb04_cmpl_seq = tuple(
    os.path.join(idl_cb04, pkg + ".pro")
    for pkg in idl_cb04_subpkg
); 

In [12]:
idlpy.IDL.e = idlpy.IDL.envi(headless=False); 

% Restored file: ENVI.
% Loaded DLM: HPGRAPHICS.
% Compiled module: ENVI_VECTOR_MASK_RASTER_CLASSIC.
% Loaded DLM: PNG.
% Loaded DLM: URL.


In [13]:
for pkg in idl_cb04_cmpl_seq: 
    idlpy.IDL.run(".compile -v {pkg}".format(pkg=pkg)); 

## 影像数据读取

### 影像存放路径定位

In [14]:
rs_meta_dir = os.path.normpath(
    os.path.join(
        nb_dir, os.pardir, os.pardir, os.pardir, 
        os.pardir, "Source", "Imagery"
    )
); 

### 文件名匹配

In [15]:
p10_finder = cresda.cb04.P10(); 
p10_finder.source_dir = rs_meta_dir; 
p10_finder.target_dir = rs_meta_dir; 

In [16]:
cb04_p10_scenes = tuple(scene for scene in p10_finder.traverse()); 

## 元数据持久化

In [17]:
for scene in cb04_p10_scenes: 
    output_uri = os.path.join(
        nb_dir, cb04_p10_scenes[0].groupdict["archive"] + "_UTM49N.dat"
    ); 
    if os.path.isfile(output_uri): 
        continue; 
    #读取影像元数据xml文件位置
    xml_path = scene.target[0]; 
    #将元数据持久化为 ENVI hdr 元数据文件
    idlpy.IDL.cb04_p10_metadata_acq(xml_path); 
idlpy.IDL.e.close(); 

% Loaded DLM: NATIVE.
% Loaded DLM: JPEG2000.
% Loaded DLM: JPEG.
% Loaded DLM: HDF5.
% Loaded DLM: MAP_PE.


## 重投影
* 无法直接通过`idlpy`交互
* 已知问题: 
    * `Envi_Convert_File_Map_Projection`在`idlpy.IDL`中调用会卡死
    * 卡死现象在`idlpy.IDL`发起的`ENVI Classic`会话或`ENVI 5.x`会话下均发生
    * 机制尚未探明
* 对策: 直接在IDL命令行界面中执行
    * 需要通过`subprocess`发起子进程, 用管道连接其`stdin`; 
    * 在python中向子进程`stdin`发送`idl`命令, 以`\n`结束
    
> 直接通过子进程brutally干涉`idl`解释器命令行会话, 这是最后的选择, 代码需要从底层写起, 可读性注定不及调用`idlpy`; 此外, 不加sanitization地拼接`idl`命令, 可能存在注入漏洞

### 初始化`IDL`命令行会话子进程, 并连接其输入流

In [18]:
import subprocess as sproc; 

In [19]:
idl_interp_path = os.path.normpath(os.path.join(
    idlpy.__file__, os.pardir, os.pardir, os.pardir, 
    "bin", "bin.x86_64", "idl"
) ); 

In [20]:
idl_interp_sproc = sproc.Popen(
    [idl_interp_path], 
    stdin=sproc.PIPE, stdout=sproc.PIPE, stderr=sproc.STDOUT, 
    universal_newlines=True
); 
idl_interp_sproc.stdin.write("\x03"); 

In [21]:
idl_interp_sproc.stdin.write("e = Envi(/HEADLESS)\n"); 
idl_interp_sproc.stdin.write("\x03"); 

### 在`IDL`命令行会话中加载自定义程序包

In [22]:
idl_cb04_subpkg = (
    "reproj", 
); 
idl_cb04_cmpl_seq = tuple(
    os.path.join(idl_cb04, pkg + ".pro")
    for pkg in idl_cb04_subpkg
); 

In [23]:
for pkg in idl_cb04_cmpl_seq: 
    idl_interp_sproc.stdin.write(u".compile -v {pkg}\n".format(pkg=pkg)); 
    idl_interp_sproc.stdin.write("\x03"); 

### 调用自定义程序包功能, 完成重投影

In [24]:
for scene in cb04_p10_scenes: 
    output_uri = os.path.join(
        nb_dir, cb04_p10_scenes[0].groupdict["archive"] + "_UTM49N.dat"
    ); 
    if os.path.isfile(output_uri): 
        continue; 
    #读取影像tif文件位置
    tif_path = scene.source; 
    idl_interp_sproc.stdin.write(
        u"CB04_P10_REPROJ, " \
        u"'{uri_input!s}', '{uri_output!s}', " \
        u"TYPE=42, EPSG=32649\n".format(
            uri_input=tif_path, uri_output=output_uri
    ) ); 
    idl_interp_sproc.stdin.write("\x03"); 
    

In [25]:
idl_interp_sproc.terminate(); 